In [2]:
# 기본 라이브러리
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import plotly.express as px
import os
import ast


os.chdir("~/Workspace/Y")
os.environ["OPENAI_API_KEY"] = "sk-..."  # 실제 키 입력

In [11]:
import pandas as pd

df = pd.read_csv("logic/results/embeddings.csv")
df.head()

,text,학수번호,분반,embedding
0,프로그래밍과 문제해결,GED6011,1.0,"[-0.024946624413132668, 0.040936291217803955, ..."
1,배터리속 소재 이야기,GED6010,1.0,"[0.003713945159688592, 0.012014275416731834, -..."
2,미디어 글쓰기,COM2241,1.0,"[0.03798713907599449, 0.03360073268413544, -0...."
3,뉴스취재보도론,COM2242,1.0,"[0.03333742171525955, 0.05497995764017105, 0.0..."
4,전자회로 1,EEC2108,6.0,"[-0.005150322802364826, 0.02363019995391369, -..."


In [ ]:
import numpy as np
import ast

# user_courses: 수강한 학수번호 리스트
user_courses = [
    "GEB1112", "GEB1114", "GEB1116", "GEB1126", "GEB1143",
    "GEB1151", "GED4009", "GED2016", "GED5003", "GEE4007", "GEE4026"
]

# 1. embedding 문자열을 리스트로 변환 (필요할 경우)
df["embedding"] = df["embedding"].apply(ast.literal_eval)

# 2. 중복 제거 및 수강 과목 필터링
filtered = df[df["학수번호"].isin(set(user_courses))].drop_duplicates(subset=["학수번호"])

# 3. 임베딩 평균 계산
user_embedding = np.mean(np.vstack(filtered["embedding"].values), axis=0).astype("float32")

((1536,),
 array([-0.00168461,  0.04330964,  0.00505122, -0.00607339,  0.01678411],
       dtype=float32))

In [29]:
def visualize_clusters_interactive(csv_path, n_clusters=10, max_points_per_cluster=30):
    """
    Plotly를 이용해 CSV 임베딩의 KMeans 클러스터를 3D로 시각화.
    각 클러스터별 최대 30개만 샘플링하고, hover 시 text(문장/과목명) 표시.
    """
    # 1. CSV 불러오기
    df = pd.read_csv(csv_path)

    # 결측치 제거
    df = df[df["text"] != "담당교수명"]
    df = df[df["학수번호"].str.startswith("GE", na=False)]

    # 2. embedding 컬럼 문자열을 실제 벡터로 변환
    df["embedding"] = df["embedding"].apply(ast.literal_eval)
    embeddings = np.vstack(df["embedding"].values).astype("float32")

    # 3. KMeans 클러스터링
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = kmeans.fit_predict(embeddings)

    # 4. 각 클러스터별 최대 30개만 샘플링
    sampled_indices = []
    for c in range(n_clusters):
        cluster_idx = np.where(labels == c)[0]
        if len(cluster_idx) > max_points_per_cluster:
            cluster_idx = np.random.choice(cluster_idx, max_points_per_cluster, replace=False)
        sampled_indices.extend(cluster_idx)

    sampled_indices = np.array(sampled_indices)
    sampled_embeddings = embeddings[sampled_indices]
    sampled_labels = labels[sampled_indices]
    sampled_texts = df["text"].iloc[sampled_indices].values

    # 5. PCA 3차원 축소
    pca = PCA(n_components=3)
    reduced = pca.fit_transform(sampled_embeddings)

    # 6. Plotly 3D 시각화
    vis_df = pd.DataFrame({
        "PCA1": reduced[:, 0],
        "PCA2": reduced[:, 1],
        "PCA3": reduced[:, 2],
        "Cluster": sampled_labels,
        "text": sampled_texts,
    })

    fig = px.scatter_3d(
        vis_df,
        x="PCA1", y="PCA2", z="PCA3",
        color="Cluster",
        hover_data=["text"],
        title=f"KMeans Clusters (k={n_clusters}) - 3D 시각화 (최대 {max_points_per_cluster}개/클러스터)",
        opacity=0.8,
    )
    fig.update_traces(marker=dict(size=5))

    user_pca = pca.transform(user_embedding.reshape(1, -1))  # 기존 PCA 모델로 변환

    fig.add_scatter3d(
        x=[user_pca[0, 0]],
        y=[user_pca[0, 1]],
        z=[user_pca[0, 2]],
        mode="markers",
        marker=dict(size=8, color="red", symbol="diamond"),
        name="User Profile",
    )
    fig.show()

In [31]:
visualize_clusters_interactive("logic/results/embeddings.csv", n_clusters=10, max_points_per_cluster=30)

In [ ]:
from sklearn.manifold import TSNE
import plotly.express as px
import pandas as pd
import numpy as np
import ast  # 추가!


def visualize_tsne_3d(csv_path, n_components=3, perplexity=30, random_state=42, max_points=1000):
    """
    t-SNE로 3D 시각화 (Plotly)
    embeddings: np.ndarray (N, D)
    texts: 리스트, hover 시 표시할 문장들
    """
    # 1. CSV 불러오기
    df = pd.read_csv(csv_path)

    # 결측치 및 불필요 행 제거
    df = df[df["text"] != "담당교수명"]
    df = df[df["학수번호"].str.startswith("GE", na=False)]
    df = df.dropna(subset=["embedding"])

    # 2. embedding 문자열을 실제 벡터로 변환
    df["embedding"] = df["embedding"].apply(ast.literal_eval)
    embeddings = np.vstack(df["embedding"].values).astype("float32")

    # (선택) 너무 많으면 샘플링 (t-SNE는 계산량이 매우 큼)
    if len(embeddings) > max_points:
        idx = np.random.choice(len(embeddings), max_points, replace=False)
        embeddings = embeddings[idx]
        df = df.iloc[idx].reset_index(drop=True)

    # 3. t-SNE 차원 축소
    tsne = TSNE(n_components=n_components, perplexity=perplexity, random_state=random_state)
    reduced = tsne.fit_transform(embeddings)

    # 4. 시각화용 데이터프레임 구성
    vis_df = pd.DataFrame({
        "TSNE1": reduced[:, 0],
        "TSNE2": reduced[:, 1],
        "TSNE3": reduced[:, 2],
        "text": df["text"].values,
    })

    # 5. Plotly 시각화
    fig = px.scatter_3d(
        vis_df,
        x="TSNE1", y="TSNE2", z="TSNE3",
        hover_data=["text"],
        title=f"t-SNE 3D 시각화 (perplexity={perplexity}, 샘플={len(vis_df)})",
        opacity=0.8,
    )
    fig.update_traces(marker=dict(size=5))
    fig.show()

In [34]:
visualize_tsne_3d("logic/results/embeddings.csv", n_components=3, perplexity=30, random_state=42)

In [35]:
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import plotly.express as px
import pandas as pd
import numpy as np
import ast

def visualize_tsne_clusters(csv_path, n_components=3, perplexity=30, n_clusters=10, random_state=42, max_points=1000):
    # 1. CSV 불러오기 및 전처리
    df = pd.read_csv(csv_path)
    df = df[df["text"] != "담당교수명"]
    df = df[df["학수번호"].str.startswith("GE", na=False)]
    df = df.dropna(subset=["embedding"])
    df["embedding"] = df["embedding"].apply(ast.literal_eval)
    embeddings = np.vstack(df["embedding"].values).astype("float32")

    # 샘플링 (너무 많을 경우)
    if len(embeddings) > max_points:
        idx = np.random.choice(len(embeddings), max_points, replace=False)
        embeddings = embeddings[idx]
        df = df.iloc[idx].reset_index(drop=True)

    # 2. t-SNE 차원 축소
    tsne = TSNE(n_components=n_components, perplexity=perplexity, random_state=random_state)
    reduced = tsne.fit_transform(embeddings)

    # 3. KMeans로 군집화
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
    cluster_labels = kmeans.fit_predict(reduced)

    # 4. 시각화용 데이터프레임
    vis_df = pd.DataFrame({
        "TSNE1": reduced[:, 0],
        "TSNE2": reduced[:, 1],
        "TSNE3": reduced[:, 2],
        "Cluster": cluster_labels,
        "text": df["text"].values,
    })

    # 5. Plotly 시각화
    fig = px.scatter_3d(
        vis_df,
        x="TSNE1", y="TSNE2", z="TSNE3",
        color="Cluster",
        hover_data=["text"],
        title=f"t-SNE + KMeans Clusters (perplexity={perplexity}, k={n_clusters})",
        opacity=0.8,
    )
    fig.update_traces(marker=dict(size=5))
    fig.show()

In [37]:
visualize_tsne_clusters("logic/results/embeddings.csv", perplexity=30, n_clusters=8)

In [10]:
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import plotly.express as px
import pandas as pd
import numpy as np
import ast
import plotly.io as pio


def visualize_tsne_clusters_html(
    csv_path,
    n_components=3,
    perplexity=30,
    n_clusters=10,
    random_state=42,
    max_points=1000,
    output_html="tsne_clusters.html",
):
    """
    t-SNE + KMeans 3D 시각화 후 HTML로 저장 (브라우저에서 Ctrl + / - 확대 가능)
    """
    # 1. CSV 로드 및 전처리
    df = pd.read_csv(csv_path)
    df = df[df["text"] != "담당교수명"]
    df = df[df["학수번호"].str.startswith("GE", na=False)]
    df = df.dropna(subset=["embedding"])
    df["embedding"] = df["embedding"].apply(ast.literal_eval)
    embeddings = np.vstack(df["embedding"].values).astype("float32")

    # (선택) 데이터가 너무 많을 경우 샘플링
    if len(embeddings) > max_points:
        idx = np.random.choice(len(embeddings), max_points, replace=False)
        embeddings = embeddings[idx]
        df = df.iloc[idx].reset_index(drop=True)

    # 2. t-SNE 차원 축소
    tsne = TSNE(n_components=n_components, perplexity=perplexity, random_state=random_state, n_iter=2000)
    reduced = tsne.fit_transform(embeddings)

    # 3. KMeans 군집화
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
    cluster_labels = kmeans.fit_predict(reduced)

    # 4. 시각화용 데이터프레임 생성
    vis_df = pd.DataFrame({
        "TSNE1": reduced[:, 0],
        "TSNE2": reduced[:, 1],
        "TSNE3": reduced[:, 2],
        "Cluster": cluster_labels,
        "text": df["text"].values,
    })

    # 5. Plotly 시각화
    fig = px.scatter_3d(
        vis_df,
        x="TSNE1", y="TSNE2", z="TSNE3",
        color="Cluster",
        hover_data=["text"],
        title=f"t-SNE + KMeans Clusters (perplexity={perplexity}, k={n_clusters})",
        opacity=0.8,
    )
    fig.update_traces(marker=dict(size=5))

    # 6. HTML로 내보내기 (Ctrl + / - 확대 가능)
    pio.write_html(fig, file=output_html, include_plotlyjs="cdn", auto_open=True)

    print(f"✅ 시각화 HTML 파일이 생성되었습니다: {output_html}")

In [11]:
visualize_tsne_clusters_html(
    "logic/results/embeddings.csv",
    perplexity=35,
    n_clusters=8,
    max_points=1500,
    output_html="tsne_kmeans_cluster_view.html"
)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning:

'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.



✅ 시각화 HTML 파일이 생성되었습니다: tsne_kmeans_cluster_view.html


In [3]:
    # ===============================
    # BERT 임베딩 CSV 로드 + t-SNE + KMeans
    # ===============================
    import pandas as pd
    import numpy as np
    from sklearn.manifold import TSNE
    from sklearn.cluster import KMeans
    import plotly.express as px
    import ast


    # -------------------------------
    # 1. CSV에서 임베딩 로드 함수
    # -------------------------------
    def load_bert_embeddings(csv_path: str):
        df = pd.read_csv(csv_path)
        # 'embedding' 컬럼 문자열 → 리스트 변환
        df["embedding"] = df["embedding"].apply(ast.literal_eval)
        embeddings = np.vstack(df["embedding"].values)
        texts = df["교과목명"].values
        return embeddings, texts


    # -------------------------------
    # 2. t-SNE + KMeans + 시각화
    # -------------------------------
    def visualize_tsne_clusters_from_csv(
        csv_path: str,
        n_components=3,
        perplexity=30,
        n_clusters=10,
        random_state=42,
        max_points=1000,
    ):
        # (1) 임베딩 불러오기
        embeddings, texts = load_bert_embeddings(csv_path)

        # (2) 샘플링
        if len(embeddings) > max_points:
            idx = np.random.choice(len(embeddings), max_points, replace=False)
            embeddings = embeddings[idx]
            texts = texts[idx]

        # (3) t-SNE 차원 축소
        tsne = TSNE(
            n_components=n_components, perplexity=perplexity, random_state=random_state
        )
        reduced = tsne.fit_transform(embeddings)

        # (4) KMeans 군집화
        kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
        cluster_labels = kmeans.fit_predict(reduced)

        # (5) DataFrame 구성
        vis_df = pd.DataFrame({
            "TSNE1": reduced[:, 0],
            "TSNE2": reduced[:, 1],
            "TSNE3": reduced[:, 2],
            "Cluster": cluster_labels,
            "text": texts,
        })

        # (6) 3D 시각화
        fig = px.scatter_3d(
            vis_df,
            x="TSNE1", y="TSNE2", z="TSNE3",
            color="Cluster",
            hover_data=["text"],
            title=f"BERT Embeddings t-SNE + KMeans (perplexity={perplexity}, k={n_clusters})",
            opacity=0.8,
        )
        fig.update_traces(marker=dict(size=5))
        fig.show()

        return vis_df




In [ ]:
vis_df = visualize_tsne_clusters_from_csv(
    csv_path="logic/results/bert_embeddings.csv",
    perplexity=40,
    n_clusters=10,
    max_points=800,
)

In [8]:
import pandas as pd
import ast

df = pd.read_csv("logic/results/bert_embeddings.csv")
embedding = ast.literal_eval(df.iloc[0]['embedding'])
print(type(embedding), len(embedding))

<class 'list'> 384


In [ ]:
    # ===============================
    # BERT 임베딩 CSV 로드 + t-SNE + KMeans
    # ===============================
    import pandas as pd
    import numpy as np
    from sklearn.manifold import TSNE
    from sklearn.cluster import KMeans
    import plotly.express as px
    import ast


    # -------------------------------
    # 1. CSV에서 임베딩 로드 함수
    # -------------------------------
    def load_bert_embeddings(csv_path: str):
        df = pd.read_csv(csv_path)
        df = df[df["학수번호"].str.startswith("GE", na=False)]
        # 'embedding' 컬럼 문자열 → 리스트 변환
        df["embedding"] = df["embedding"].apply(ast.literal_eval)
        embeddings = np.vstack(df["embedding"].values)
        texts = df["교과목명"].values
        return embeddings, texts


    # -------------------------------
    # 2. t-SNE + KMeans + 시각화
    # -------------------------------
    def visualize_tsne_clusters_from_csv(
        csv_path: str,
        n_components=3,
        perplexity=30,
        n_clusters=10,
        random_state=42,
        max_points=1000,
    ):
        # (1) 임베딩 불러오기
        embeddings, texts = load_bert_embeddings(csv_path)

        # (2) 샘플링
        if len(embeddings) > max_points:
            idx = np.random.choice(len(embeddings), max_points, replace=False)
            embeddings = embeddings[idx]
            texts = texts[idx]

        # (3) t-SNE 차원 축소
        tsne = TSNE(
            n_components=n_components, perplexity=perplexity, random_state=random_state
        )
        reduced = tsne.fit_transform(embeddings)

        # (4) KMeans 군집화
        kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
        cluster_labels = kmeans.fit_predict(reduced)

        # (5) DataFrame 구성
        vis_df = pd.DataFrame({
            "TSNE1": reduced[:, 0],
            "TSNE2": reduced[:, 1],
            "TSNE3": reduced[:, 2],
            "Cluster": cluster_labels,
            "text": texts,
        })

        # (6) 3D 시각화
        fig = px.scatter_3d(
            vis_df,
            x="TSNE1", y="TSNE2", z="TSNE3",
            color="Cluster",
            hover_data=["text"],
            title=f"BERT Embeddings t-SNE + KMeans (perplexity={perplexity}, k={n_clusters})",
            opacity=0.8,
        )
        fig.update_traces(marker=dict(size=5))
        fig.show()

        return vis_df


In [7]:
vis_df = visualize_tsne_clusters_from_csv(
    csv_path="logic/results/bert_embeddings.csv",
    perplexity=40,
    n_clusters=10,
    max_points=800,
)

In [8]:
# 1️⃣ 임베딩 CSV 불러오기
df = pd.read_csv("logic/results/bert_embeddings.csv")

# 2️⃣ 문자열로 저장된 벡터를 리스트로 변환
df["embedding"] = df["embedding"].apply(ast.literal_eval)

# 3️⃣ numpy 배열로 변환
import numpy as np

X = np.vstack(df["embedding"].values)

# 4️⃣ KMeans 클러스터링 (예: 10개 클러스터)
n_clusters = 10
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
labels = kmeans.fit_predict(X)
df["cluster"] = labels

# 5️⃣ 클러스터별 10개씩 출력
for i in range(n_clusters):
    subset = df[df["cluster"] == i].head(10)
    print(f"\n=== 클러스터 {i} ===")
    print(subset[["학수번호", "분반", "교과목명"]].to_string(index=False))


=== 클러스터 0 ===
   학수번호  분반     교과목명
ACE1901 1.0 정보사회와컴퓨터
ACE1903 1.0     지구과학
ACE1904 1.0     생명과학
ACE1906 1.0      C언어
ACE1906 4.0      C언어
ACE1906 6.0      C언어
ACE2901 1.0   공업수학 1
ACE2902 1.0   공업수학 2
ACE2902 2.0   공업수학 2
ACE2902 6.0   공업수학 2

=== 클러스터 1 ===
   학수번호  분반             교과목명
ABC9008 1.0        바이오산업현장실무
ABC9009 1.0        바이오산학프로젝트
ABC9010 1.0 바이오공정제어를위한물리화학기초
ABC9011 1.0    바이오의약품스마트생산설계
ACE1904 3.0             생명과학
ASE1010 1.0              정역학
ASE3113 1.0       항공우주공학실험 2
ASE3132 1.0           저속공기역학
AUS3132 1.0        항공드론 기초역학
BIO1902 1.0            생명과학2

=== 클러스터 2 ===
   학수번호  분반          교과목명
AAO3034 1.0           북한학
AAO3035 1.0        세계 전쟁사
CCM3902 1.0 한국문화의 콘텐츠화 실습
CHN1008 1.0      중국학 입문 2
CHN1008 2.0      중국학 입문 2
CHN1012 1.0         중국어 2
CHN1012 3.0         중국어 2
CHN1012 4.0         중국어 2
CHN2012 1.0       집중중국어 2
CHN2012 2.0       집중중국어 2

=== 클러스터 3 ===
   학수번호  분반     교과목명
UMD1025 1.0   종합평가 2
UMD2025 1.0   종합평가 3
UMD3037 1.0 정신건강의학실습
    NaN NaN    담

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# 1️⃣ 임베딩 모델 정의
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 2️⃣ FAISS 인덱스 불러오기
db = FAISS.load_local(
    "logic/results/faiss_index", 
    embedding_model, 
    allow_dangerous_deserialization=True
)

# 3️⃣ 차원 수 확인
index = db.index
print("임베딩 차원 수:", index.d)

임베딩 차원 수: 1536


In [19]:
def visualize_tsne_clusters(csv_path, n_components=3, perplexity=30, n_clusters=10, random_state=42, max_points=1000):
    from sklearn.manifold import TSNE
    from sklearn.cluster import KMeans
    import plotly.express as px
    import pandas as pd
    import numpy as np
    import ast

    # 1. CSV 불러오기 및 전처리
    df = pd.read_csv(csv_path)
    df = df[df["text"] != "담당교수명"]
    df = df[df["학수번호"].str.startswith("GE", na=False)]
    df = df.dropna(subset=["embedding"])
    df["embedding"] = df["embedding"].apply(ast.literal_eval)
    embeddings = np.vstack(df["embedding"].values).astype("float32")

    # 샘플링
    if len(embeddings) > max_points:
        idx = np.random.choice(len(embeddings), max_points, replace=False)
        embeddings = embeddings[idx]
        df = df.iloc[idx].reset_index(drop=True)

    # 2. t-SNE
    tsne = TSNE(n_components=n_components, perplexity=perplexity, random_state=random_state)
    reduced = tsne.fit_transform(embeddings)

    # 3. KMeans
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
    cluster_labels = kmeans.fit_predict(reduced)

    # 4. 시각화용 데이터프레임
    vis_df = pd.DataFrame({
        "TSNE1": reduced[:, 0],
        "TSNE2": reduced[:, 1],
        "TSNE3": reduced[:, 2],
        "Cluster": cluster_labels,
        "text": df["text"].values,
        "학수번호": df["학수번호"].values
    })

    # 5. 클러스터별 10개씩 출력
    print("\n=== 각 클러스터별 대표 10개 ===")
    for c in range(n_clusters):
        sample = vis_df[vis_df["Cluster"] == c].sample(min(10, len(vis_df[vis_df["Cluster"] == c])), random_state=random_state)
        print(f"\n[Cluster {c}]")
        print(sample[["학수번호", "text"]].to_string(index=False))

    # 6. 3D 시각화
    fig = px.scatter_3d(
        vis_df,
        x="TSNE1", y="TSNE2", z="TSNE3",
        color="Cluster",
        hover_data=["학수번호", "text"],
        title=f"t-SNE + KMeans Clusters (perplexity={perplexity}, k={n_clusters})",
        opacity=0.8,
    )
    fig.update_traces(marker=dict(size=5))
    fig.show()

    return vis_df

In [21]:
visualize_tsne_clusters(
    "logic/results/embeddings.csv",
    perplexity=40,
    n_clusters=10,
    max_points=800,
)


=== 각 클러스터별 대표 10개 ===

[Cluster 0]
   학수번호             text
GED4009        동북아와 한일관계
GEB1112 크로스오버 1 : 인간의 탐색
GEB1118 우리가 만드는 세계(WWM1)
GED2016        그리스신화의 이해
GEB1118 우리가 만드는 세계(WWM1)
GEB1119 우리가 만나는 세계(WWM2)
GEE2040      영화로 보는 생명공학
GEB1119 우리가 만나는 세계(WWM2)
GED4006          국제관계학개론
GEB1119 우리가 만나는 세계(WWM2)

[Cluster 1]
   학수번호             text
GEB1114 크로스오버 3 : 사회의 탐색
GEB1114 크로스오버 3 : 사회의 탐색
GEB1114 크로스오버 3 : 사회의 탐색
GEE2023       뉴스 속의 법이야기
GEE2005          경제학의 이해
GEB1113 크로스오버 2 : 자연의 탐색
GEE2041 차세대 해양리더 양성 아카데미
GED2010         축제와 인간사회
GEB1114 크로스오버 3 : 사회의 탐색
GED1002         공학윤리와 토론

[Cluster 2]
   학수번호               text
GEB1108        의사소통 영어: 중급
GEB1107            의사소통 영어
GEB1117          커리어 디자인 1
GEB1108        의사소통 영어: 중급
GEB1109        의사소통 영어: 고급
GEB1107            의사소통 영어
GEB1107            의사소통 영어
GEE3028           실용영어 R/W
GEE3001           졸업인증실용영어
GED1007 독서세미나 : 인간, 가치, 공존

[Cluster 3]
   학수번호         text
GEB1126 문제해결을 위한 글쓰기
GEB1126 문제해결을 위한 글쓰기
GEB1126 문제해결을 

,TSNE1,TSNE2,TSNE3,Cluster,text,학수번호
0,49.716511,-7.996064,30.456968,7,프로그래밍과 문제해결,GED6011
1,69.394760,1.078883,-3.241318,8,배터리속 소재 이야기,GED6010
2,-46.071602,26.886621,-19.667835,6,외국인전용 한국의 전통 사회와 문화,GEF1103
3,-40.593723,-12.708159,-18.221952,6,외국인전용 대학 한국어Ⅰ,GEF1105
4,-34.460361,15.356696,-22.066893,6,외국인전용 한국의 현대 사회와 문화,GEF1104
...,...,...,...,...,...,...
259,26.782520,-15.090016,52.244732,7,공간정보종합설계,GEO4010
260,23.764786,50.349495,31.575090,1,공학윤리와 토론,GED1002
261,33.304375,51.883167,24.571630,1,공학윤리와 토론,GED1002
262,24.869444,43.436440,21.891014,1,공학윤리와 토론,GED1002
